# AV stage-cost model (offline)

This notebook calculates with the same model.py used by the command line.
The checked-in receipt scenario includes completed ASR and direct-video baseline
attempts, but no completed AV Grok+Jev comparison. Measured tokens, list-rate
estimates, unknown costs, failures, and reservation states remain separate. Only
retained reservations count against current cap headroom; released and superseded
ceilings remain visible as history. See README.md for provenance and limitations.
These cells never call a provider or fetch media.


In [1]:
import json
import sys
from pathlib import Path

root = Path.cwd()
recipe = root if (root / "model.py").is_file() else root / "cookbook" / "cost-model"
sys.path.insert(0, str(recipe.resolve()))
from model import experiment_report, jsonable, scenario, token_cost

input_path = recipe / "scenario.receipts.json"
inputs = json.loads(input_path.read_text())
print(json.dumps(inputs["experiment"], indent=2))
accounting = experiment_report(inputs)
print(json.dumps(jsonable({
    "measured_usage": accounting["measured_usage"],
    "known_list_estimates_usd": accounting["list_rate_estimates"]["known_subtotal_usd"],
    "unknown_costs": accounting["unknown_costs"],
    "failures": accounting["failures"],
    "reservations": accounting["reservations"],
    "cumulative_cap_usd": accounting["cumulative_cap_usd"],
    "known_estimates_plus_retained_usd": accounting["known_estimates_plus_retained_usd"],
    "remaining_cap_usd": accounting["remaining_cap_usd"],
}), indent=2))


{
  "status": "partial_reproduction",
  "implementation": "av",
  "receipt_directory": "../receipts",
  "note": "Completed ASR and direct-video baseline receipts exist. Two exact-model 32-token compatibility probes completed with usage but ignored the requested output cap. A later fresh attempt at AV commit e24845d imported the transcript, then stopped on a local media-probe timeout before frame extraction with zero provider requests. No completed AV Grok+Jev comparison exists yet.",
  "projection": "Repeated-query totals are projections from entered stage costs, not additional measured runs."
}
{
  "measured_usage": [
    {
      "name": "completed_asr",
      "input_tokens": 118228,
      "output_tokens": 13241,
      "thinking_tokens": null,
      "cached_tokens": null,
      "requests": 75,
      "outcome": "completed",
      "receipt": "../receipts/asr.json",
      "note": "Complete source audio submitted in fixed windows; transcript accuracy and word completeness were not verifie

## Compare query volumes without hiding unknowns

Indexing is charged once. Retrieval, judge, answer, and optional stronger
inspection are multiplied by query volume. Hosting and storage refer to the
same observation period. A complete modeled total can contain assumptions; it
is not necessarily a measured bill. This receipt scenario remains incomplete
because no completed AV Grok+Jev query exists. Repeated-query projections are
not new benchmark measurements.


In [2]:
for queries in (1, 100, 1000):
    result = scenario(inputs, queries)
    print(json.dumps(jsonable({
        "queries": queries,
        "known_subtotal_usd": result["indexed"]["known_subtotal_usd"],
        "complete_total_usd": result["indexed"]["complete_total_usd"],
        "unknown_terms": result["indexed"]["unknown_terms"],
        "uncached_to_indexed_before_unknown_costs_ratio": result["uncached_to_indexed_before_unknown_costs_ratio"],
    }), indent=2))


{
  "queries": 1,
  "known_subtotal_usd": "0.0685709",
  "complete_total_usd": null,
  "unknown_terms": [
    "indexing.0.caption",
    "indexing.3.preprocessing",
    "query.retrieval",
    "query.judge",
    "query.answer",
    "period.hosting",
    "period.storage"
  ],
  "uncached_to_indexed_before_unknown_costs_ratio": null
}
{
  "queries": 100,
  "known_subtotal_usd": "0.0685709",
  "complete_total_usd": null,
  "unknown_terms": [
    "indexing.0.caption",
    "indexing.3.preprocessing",
    "query.retrieval",
    "query.judge",
    "query.answer",
    "period.hosting",
    "period.storage"
  ],
  "uncached_to_indexed_before_unknown_costs_ratio": null
}
{
  "queries": 1000,
  "known_subtotal_usd": "0.0685709",
  "complete_total_usd": null,
  "unknown_terms": [
    "indexing.0.caption",
    "indexing.3.preprocessing",
    "query.retrieval",
    "query.judge",
    "query.answer",
    "period.hosting",
    "period.storage"
  ],
  "uncached_to_indexed_before_unknown_costs_ratio": nul

## Edit cache and fallback assumptions explicitly

In the input JSON, record cache policy (TTL, reuse scope, refresh count and
storage duration), hit rate, all-in cache-hit request cost, writes and storage.
Reuse can span requests and users of one application. Do not infer a universal
cache policy from one request with zero cached tokens. The stronger fallback
frequency and per-invocation price are separate inputs; a missing price stays
unknown when the fallback runs. Historical Composer inputs are not AV evidence.


In [3]:
result = scenario(inputs, 100)
print(json.dumps(jsonable({
    "cache_policy": result["cache_policy"],
    "cache_hit_rate": result["cache_hit_rate"],
    "cache_baseline": result["cache_policy_baseline"],
    "fallback": inputs["query"]["stronger_fallback"],
}), indent=2))


{
  "cache_policy": "disabled for the single recorded baseline request; no reusable cache was explicitly created",
  "cache_hit_rate": "0",
  "cache_baseline": {
    "terms": [
      {
        "name": "baseline.cache_misses",
        "usd": "30.807600",
        "basis": "estimated",
        "note": "One completed Gemini 3.8 native-video query: measured provider tokens multiplied by recorded upstream list rates; actual billing is unknown."
      },
      {
        "name": "baseline.cache_hits",
        "usd": "0",
        "basis": "unknown",
        "note": "No cache-hit request was measured."
      },
      {
        "name": "baseline.cache_writes",
        "usd": "0",
        "basis": "not_used",
        "note": "No explicit cache write was requested."
      },
      {
        "name": "baseline.cache_storage",
        "usd": "0",
        "basis": "not_used",
        "note": "No explicit cache storage was requested."
      }
    ],
    "known_subtotal_usd": "30.807600",
    "unknown_te